# KaraokeForge on Kaggle GPU

Run this notebook in a Kaggle Notebook with **GPU** enabled. It starts the real `worker/` service from the GitHub repo and exposes it with a free Cloudflare Quick Tunnel.

The notebook uses Kaggle's remote GPU for Demucs/Whisper. Your local computer does not perform the AI processing.

In [ ]:
!rm -rf /kaggle/working/KaraokeForge
!git clone -q https://github.com/omaparekh-ux/KaraokeForge.git /kaggle/working/KaraokeForge
%cd /kaggle/working/KaraokeForge/worker


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -r requirements.txt


In [ ]:
import os
os.environ["JOBS_DIR"] = "/kaggle/working/karaoke_jobs"
os.environ["JOB_DB"] = "/kaggle/working/karaoke_jobs/karaokeforge.db"
os.environ["FRONTEND_ORIGINS"] = "*"
os.environ["DEMUCS_DEVICE"] = "cuda"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE"] = "float16"
os.environ["WHISPER_MODEL"] = "small"
os.environ["DEMUCS_MODEL"] = "htdemucs"
os.environ["MAX_CONCURRENT_JOBS"] = "1"
os.environ["MAX_UPLOAD_MB"] = "250"
print("Configured KaraokeForge worker")


In [ ]:
!nvidia-smi -L


In [ ]:
import subprocess, time, os
log_path = "/kaggle/working/karaokeforge-worker.log"
log = open(log_path, "w")
proc = subprocess.Popen(["python","-m","uvicorn","app:app","--host","0.0.0.0","--port","8000"], cwd="/kaggle/working/KaraokeForge/worker", env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
time.sleep(4)
print("Worker PID:", proc.pid)
print(open(log_path).read()[-2000:])


In [ ]:
import subprocess, time, re, os
from pathlib import Path
cloudflared = "/kaggle/working/cloudflared"
log2_path = "/kaggle/working/cloudflared.log"
if not os.path.exists(cloudflared):
    subprocess.run(["bash","-lc","curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared"], check=True)
try:
    tunnel
except NameError:
    tunnel = None
if tunnel is None or tunnel.poll() is not None:
    log2 = open(log2_path, "w")
    tunnel = subprocess.Popen([cloudflared, "tunnel", "--url", "http://127.0.0.1:8000"], stdout=log2, stderr=subprocess.STDOUT)
def get_tunnel_url():
    try:
        text = Path(log2_path).read_text(errors="ignore")
    except FileNotFoundError:
        return None
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", text)
    return match.group(0) if match else None
WORKER_URL = None
for _ in range(30):
    WORKER_URL = get_tunnel_url()
    if WORKER_URL:
        break
    time.sleep(1)
if not WORKER_URL:
    print(Path(log2_path).read_text(errors="ignore")[-5000:])
    raise RuntimeError("Cloudflare tunnel started but its URL was not found in the log. Run this cell again.")
print("KaraokeForge worker URL:")
print(WORKER_URL)


In [ ]:
import requests
health = requests.get(WORKER_URL + "/health", timeout=20)
health.raise_for_status()
print(health.json())
print("\nPaste this URL into KaraokeForge → Processing engine → Save worker:")
print(WORKER_URL)


## Keep the session alive

This is a free, temporary worker. Keep the Kaggle session running while processing songs. When the session ends, the worker and temporary Cloudflare URL disappear.

The worker itself is the same FastAPI implementation in `worker/`; the notebook only provides temporary remote GPU compute.